# Validation + Retry Loops: Pydantic / Zod

**Phase 01** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-01/07-validation-retry-loops.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic pydantic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You implement structured extraction using tool_use (from Lesson 06). For most documents, the extraction is perfect. But some documents produce outputs that pass JSON parsing yet fail your business rules: a `confidence` field that should be between 0 and 1 comes back as 87 (should be 0.87), a `status` enum contains a value not in your allowed set, or a required nested field is null when the document clearly contains the information.

Tool use guarantees valid JSON. It does not guarantee that the values meet your schema constraints, business rules, or downstream requirements. Pydantic validation catches these cases, but catching the error is only half the problem. The naive approach is to rais...

## The Concept

### The Retry Loop

```
                  ┌─────────────────────────────┐
                  │        Attempt N             │
                  │  Send prompt to model        │
                  └──────────────┬──────────────┘
                                 │ response
                                 ▼
                  ┌─────────────────────────────┐
                  │     Parse / Extract          │
                  │  JSON parse or tool_use      │
                  └──────────────┬──────────────┘
                                 │ dict
                                 ▼
                  ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 01-07: Validation + Retry Loops: Pydantic
===================================================
Demonstrates the retry-with-feedback pattern for structured extraction:

  1. Extract using tool_use
  2. Validate with Pydantic
  3. On failure: append the validation error to the conversation and retry
  4. Compare against blind retry (same prompt, no error feedback)

The key insight: feeding the validation error back to the model is 2-3x
more effective than retrying with the same prompt because the model knows
exactly which field failed and why.

Run:
    export ANTHROPIC_API_KEY=sk-ant-...
    python main.py
"""

import anthropic
import os
import json
from pydantic import BaseModel, field_validator, model_validator
from typing import Literal

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-3-5-haiku-20241022"

### Pydantic models

In [ ]:
class RiskFinding(BaseModel):
    """A single risk identified in a security assessment."""
    title: str
    severity: Literal["low", "medium", "high", "critical"]
    likelihood: float  # 0.0 to 1.0
    affected_component: str

    @field_validator("likelihood")
    @classmethod
    def likelihood_range(cls, v: float) -> float:
        if not (0.0 <= v <= 1.0):
            raise ValueError(
                f"likelihood must be between 0.0 and 1.0, got {v}. "
                "Express as a decimal (0.75 = 75%), not as a percentage (75)."
            )
        return v


class RiskAssessment(BaseModel):
    """Structured risk assessment extracted from a security report."""
    asset_name: str
    assessment_date: str
    overall_risk_score: float  # 0.0 to 1.0
    findings: list[RiskFinding]
    recommended_action: Literal["monitor", "remediate", "escalate", "accept"]
    reviewer: str

    @field_validator("overall_risk_score")
    @classmethod
    def risk_score_range(cls, v: float) -> float:
        if not (0.0 <= v <= 1.0):
            raise ValueError(
                f"overall_risk_score must be between 0.0 and 1.0, got {v}. "
                "Use decimal notation: 0.75 not 75."
            )
        return v

    @model_validator(mode="after")
    def findings_not_empty(self) -> "RiskAssessment":
        if not self.findings:
            raise ValueError("findings must contain at least one risk finding")
        return self

### Extraction tool schema

In [ ]:
RISK_SCHEMA = {
    "type": "object",
    "properties": {
        "asset_name": {"type": "string", "description": "Name of the system or asset being assessed"},
        "assessment_date": {"type": "string", "description": "Date of assessment in YYYY-MM-DD format"},
        "overall_risk_score": {
            "type": "number",
            "description": "Numeric risk score from 0.0 (no risk) to 1.0 (maximum risk). "
                           "IMPORTANT: use decimal notation - write 0.75 not 75.",
        },
        "findings": {
            "type": "array",
            "description": "List of identified risk findings",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "severity": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"],
                    },
                    "likelihood": {
                        "type": "number",
                        "description": "Likelihood from 0.0 to 1.0. Write 0.75 not 75.",
                    },
                    "affected_component": {"type": "string"},
                },
                "required": ["title", "severity", "likelihood", "affected_component"],
            },
        },
        "recommended_action": {
            "type": "string",
            "enum": ["monitor", "remediate", "escalate", "accept"],
            "description": "One of: monitor, remediate, escalate, accept",
        },
        "reviewer": {"type": "string"},
    },
    "required": [
        "asset_name", "assessment_date", "overall_risk_score",
        "findings", "recommended_action", "reviewer",
    ],
}

EXTRACTION_TOOL = {
    "name": "extract_risk_assessment",
    "description": "Extract a structured risk assessment from a security report.",
    "input_schema": RISK_SCHEMA,
}

### Sample document

In [ ]:
SAMPLE_REPORT = """
Security Assessment Report
Asset: Customer Data API v2.3
Reviewer: Alex Kim
Date: 2024-11-15

Executive Summary:
This assessment covers the Customer Data API serving approximately 50,000 requests/day.
Overall risk level is HIGH at 82 percent.

Findings:

1. SQL Injection Vulnerability in /search endpoint
   Component: Database query layer
   Likelihood: 68 percent
   Severity: CRITICAL

2. Missing rate limiting on authentication endpoints
   Component: Auth service
   Likelihood: 90 percent
   Severity: HIGH

3. Outdated TLS certificate (expires in 14 days)
   Component: Load balancer
   Likelihood: 95 percent
   Severity: MEDIUM

Recommendation: Immediate remediation required before next deployment.
"""

### Method A: Validated completion with error feedback (the right way)

In [ ]:
def validated_completion(
    document: str,
    max_retries: int = 3,
    verbose: bool = True,
) -> tuple[RiskAssessment | None, dict]:
    """
    Extract and validate a RiskAssessment from `document`.
    On Pydantic validation failure, feeds the error back to the model and retries.

    Returns:
        (result, stats) where stats contains attempt count and error history
    """
    messages = [
        {
            "role": "user",
            "content": "Extract a risk assessment from this security report:\n\n" + document,
        }
    ]

    stats = {"attempts": 0, "errors": [], "strategy": "error_feedback"}

    for attempt in range(1, max_retries + 1):
        stats["attempts"] = attempt
        if verbose:
            print(f"    Attempt {attempt}/{max_retries}...")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=[EXTRACTION_TOOL],
            tool_choice={"type": "any"},
            messages=messages,
        )

        # Extract the tool_use block
        extracted_data = None
        for block in response.content:
            if block.type == "tool_use" and block.name == "extract_risk_assessment":
                extracted_data = block.input
                break

        if extracted_data is None:
            msg = f"No tool_use block (stop_reason={response.stop_reason})"
            stats["errors"].append(msg)
            if verbose:
                print(f"    {msg}")
            break

        # Pydantic validation
        try:
            result = RiskAssessment(**extracted_data)
            if verbose:
                print(f"    Validation passed on attempt {attempt}.")
            return result, stats

        except Exception as e:
            error_msg = str(e)
            stats["errors"].append(error_msg)
            if verbose:
                print(f"    Validation failed: {error_msg[:150]}")

            if attempt == max_retries:
                if verbose:
                    print(f"    Max retries exhausted.")
                return None, stats

            # Feed the error back: append model response + user correction
            messages.append({
                "role": "assistant",
                "content": response.content,
            })
            messages.append({
                "role": "user",
                "content": (
                    f"Validation failed with this error:\n\n{error_msg}\n\n"
                    "Please correct only the fields that failed. "
                    "Remember: scores and likelihoods must be decimals (0.75), not percentages (75). "
                    "Call the tool again with the corrected values."
                ),
            })

    return None, stats

### Method B: Blind retry (the naive approach - for comparison)

In [ ]:
def blind_retry(
    document: str,
    max_retries: int = 3,
    verbose: bool = True,
) -> tuple[RiskAssessment | None, dict]:
    """
    Naive approach: retry the same prompt on failure.
    Does NOT feed validation errors back to the model.
    For comparison with validated_completion.
    """
    stats = {"attempts": 0, "errors": [], "strategy": "blind_retry"}
    base_messages = [
        {
            "role": "user",
            "content": "Extract a risk assessment from this security report:\n\n" + document,
        }
    ]

    for attempt in range(1, max_retries + 1):
        stats["attempts"] = attempt
        if verbose:
            print(f"    Attempt {attempt}/{max_retries}...")

        # Always restart from the same prompt (no error context)
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=[EXTRACTION_TOOL],
            tool_choice={"type": "any"},
            messages=base_messages,
        )

        for block in response.content:
            if block.type == "tool_use":
                try:
                    result = RiskAssessment(**block.input)
                    if verbose:
                        print(f"    Validation passed on attempt {attempt}.")
                    return result, stats
                except Exception as e:
                    error_msg = str(e)
                    stats["errors"].append(error_msg)
                    if verbose:
                        print(f"    Validation failed: {error_msg[:150]}")
                    break  # move to next attempt without any feedback

    return None, stats

### Display helpers

In [ ]:
def print_result(result: RiskAssessment | None, stats: dict) -> None:
    """Print a formatted result summary."""
    strategy = stats["strategy"]
    attempts = stats["attempts"]

    if result:
        print(f"  SUCCESS after {attempts} attempt(s) [{strategy}]")
        print(f"  Asset: {result.asset_name}")
        print(f"  Risk score: {result.overall_risk_score} (should be 0.0-1.0)")
        print(f"  Action: {result.recommended_action}")
        print(f"  Findings: {len(result.findings)}")
        for f in result.findings:
            print(f"    - [{f.severity}] {f.title} (likelihood={f.likelihood})")
    else:
        print(f"  FAILED after {attempts} attempt(s) [{strategy}]")
        print(f"  Errors:")
        for err in stats["errors"]:
            print(f"    - {err[:120]}")


# ---------------------------------------------------------------------------
# Run comparison
# ---------------------------------------------------------------------------

### Demo

In [ ]:
print("Validation + Retry Loops: Error-Feedback vs. Blind Retry")
print("=" * 65)
print(f"\nDocument: Security Assessment Report (contains % values that should be decimals)")

print("\n--- Strategy A: Error-Feedback Retry ---")
result_a, stats_a = validated_completion(SAMPLE_REPORT, max_retries=3, verbose=True)
print_result(result_a, stats_a)

print("\n--- Strategy B: Blind Retry ---")
result_b, stats_b = blind_retry(SAMPLE_REPORT, max_retries=3, verbose=True)
print_result(result_b, stats_b)

print("\n--- Summary ---")
print(f"Error-feedback: {'SUCCESS' if result_a else 'FAILED'} in {stats_a['attempts']} attempt(s)")
print(f"Blind retry:    {'SUCCESS' if result_b else 'FAILED'} in {stats_b['attempts']} attempt(s)")

---

## Self-Check

Answer these without running code first:

**1. Why does tool_use not prevent this problem, and what is the correct fix?**

- A. tool_use guarantees JSON validity but not value correctness; add a Pydantic field_validator that rejects values > 1.0 and feeds the error back to the model
- B. The problem is in the JSON Schema; change confidence_score type from number to integer
- C. tool_use is not being used correctly; use prompt-only extraction instead for numeric fields
- D. The downstream code needs to normalize the value by dividing by 100

**2. What is wrong with your retry message?**

- A. The assistant response should not be appended to the messages array
- B. The retry message 'Please try again' does not tell the model what failed or how to fix it; it needs to include the validation error text
- C. You should start a new conversation thread for each retry attempt
- D. max_retries=3 is too low; increase it to 10

**3. Which fallback strategy would have prevented this data loss while keeping the pipeline running?**

- A. Increase max_retries to 10 so fewer documents exhaust retries
- B. Return None but write the failed document to a dead-letter queue for manual review, and increment a monitoring counter
- C. Return the raw unvalidated dict from the last attempt, marked with a 'validated: false' flag, and log a warning
- D. Both B and C are reasonable fallback strategies depending on whether you need the partial data or can wait for manual review

**4. What does this result tell you about the nature of the first-attempt failures?**

- A. The model is unreliable and the results are too random to draw conclusions
- B. Most first-attempt failures are formatting errors (percentage vs. decimal, wrong enum value) that the model can correct immediately when told what went wrong
- C. The documents are genuinely ambiguous and the model needs more context, not just error feedback
- D. The 76% success rate means error-feedback has a 24% fundamental failure rate and max_retries should be increased

**5. Should you add this normalizing validator, and what is the tradeoff?**

- A. No: normalizing validators hide schema violations that should be fixed at the source (the extraction prompt/schema)
- B. Yes: it is pragmatic to normalize casing since it does not affect meaning, and it reduces unnecessary retry calls
- C. It depends: if 'Active' vs 'active' is a consistent capitalization pattern across all documents, normalize it; if it is a random model error, fix the schema description instead
- D. Yes, but only for the status field: use normalizing validators only for enum fields, never for numeric fields

**6. What is the main advantage of keeping the manual validated_completion implementation alongside instructor?**

- A. The manual version is always faster because it has less overhead
- B. The manual version gives you full visibility into each attempt's messages and errors, making it easier to debug schema and retry issues that instructor obscures
- C. instructor has bugs in its retry logic that the manual version avoids
- D. The manual version can use different models for each retry attempt

**7. What systemic change to the pipeline would reduce queue backlog without increasing manual review capacity?**

- A. Increase max_retries from 3 to 10 to reduce the number of documents that reach the dead-letter queue
- B. Audit the most common validation errors in the failed documents and improve the extraction schema and descriptions to fix the root causes
- C. Accept all None results as 'missing data' and move on without logging
- D. Switch from Pydantic validation to runtime type checking which is more permissive

**8. This 3-attempt pattern is working correctly, but you want to optimize to reduce the chance of needing a third attempt. What is the most effective change?**

- A. Run three parallel extraction calls and take the first one that validates
- B. Make the validation error message more specific: list each failing field separately with the exact value and the constraint it violated
- C. Break the schema into smaller sub-schemas and validate each independently
- D. Use a more powerful model for attempt 2 and 3 to increase the chance of full correction

_Answers are in `checks.json` in the lesson directory._